# Notebook 02 — Build Labels & Patient-Level Splits

1. Parse clinical CSV → stage labels per patient
2. Map stage strings to integer classes (I→0, II→1, III→2, IV→3)
3. Merge with `slices_index.csv`
4. **Filter to TUMOR-ONLY slices** (critical for avoiding majority-class collapse)
5. Patient-level stratified split (70/15/15)
6. Save split CSVs

In [ ]:
import os, sys
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

sys.path.insert(0, os.path.abspath('.'))

In [ ]:
# ── Configuration ────────────────────────────────────────────────
CLINICAL_CSV = '/path/to/local/resource'
SLICES_CSV   = 'processed_data/slices_index.csv'
OUTPUT_DIR   = 'processed_data'

SEED = 42
TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15
TEST_RATIO  = 0.15

# KEY SETTING: Only use slices that contain tumor
TUMOR_ONLY = True

In [ ]:
# ── A) Build stage labels per patient ────────────────────────────
clinical = pd.read_csv(CLINICAL_CSV)
print(f'Clinical data: {len(clinical)} patients')
print(f'Columns: {list(clinical.columns)}')
print(f'\nOverall.Stage value counts:')
print(clinical['Overall.Stage'].value_counts())

In [ ]:
# ── Stage mapping ────────────────────────────────────────────────
def map_stage(stage_str):
    """Map Overall.Stage string to integer class."""
    if pd.isna(stage_str) or str(stage_str).strip().upper() == 'NA':
        return np.nan
    s = str(stage_str).strip().upper()
    s = s.replace('STAGE ', '')
    
    if s in ('I', 'IA', 'IB', 'IA1', 'IA2', 'IA3', 'IB1'):
        return 0
    elif s in ('II', 'IIA', 'IIB', 'IIA1', 'IIA2', 'IIB1'):
        return 1
    elif s in ('III', 'IIIA', 'IIIB', 'IIIC'):
        return 2
    elif s.startswith('IV'):
        return 3
    else:
        print(f'  ⚠️ Unknown stage: "{stage_str}" → dropping')
        return np.nan

clinical['label'] = clinical['Overall.Stage'].apply(map_stage)

n_before = len(clinical)
clinical = clinical.dropna(subset=['label'])
clinical['label'] = clinical['label'].astype(int)
n_after = len(clinical)
print(f'\nDropped {n_before - n_after} patients with missing/unknown stage')
print(f'Remaining: {n_after} patients')
print(f'\nLabel distribution:')
stage_names = {0: 'Stage I', 1: 'Stage II', 2: 'Stage III', 3: 'Stage IV'}
for label, name in stage_names.items():
    count = (clinical['label'] == label).sum()
    if count > 0:
        print(f'  {name} (class {label}): {count}')

In [ ]:
# ── B) Patient-level stratified split ────────────────────────────
patient_labels = clinical[['PatientID', 'label']].copy()

train_pts, temp_pts = train_test_split(
    patient_labels, test_size=(VAL_RATIO + TEST_RATIO),
    stratify=patient_labels['label'], random_state=SEED
)

relative_test = TEST_RATIO / (VAL_RATIO + TEST_RATIO)
val_pts, test_pts = train_test_split(
    temp_pts, test_size=relative_test,
    stratify=temp_pts['label'], random_state=SEED
)

train_pts['split'] = 'train'
val_pts['split']   = 'val'
test_pts['split']  = 'test'

splits_df = pd.concat([train_pts, val_pts, test_pts], ignore_index=True)

print(f'Split summary:')
print(f'  Train: {len(train_pts)} patients')
print(f'  Val  : {len(val_pts)} patients')
print(f'  Test : {len(test_pts)} patients')
print(f'\nLabel distribution per split:')
print(splits_df.groupby(['split', 'label']).size().unstack(fill_value=0))

In [ ]:
# ── Save splits.csv ──────────────────────────────────────────────
splits_path = os.path.join(OUTPUT_DIR, 'splits.csv')
splits_df.to_csv(splits_path, index=False)
print(f'Saved patient-level splits to {splits_path}')

In [ ]:
# ── Merge with slice index ───────────────────────────────────────
slices_df = pd.read_csv(SLICES_CSV)
print(f'Loaded {len(slices_df)} total slices from {SLICES_CSV}')
print(f'  Tumor slices: {slices_df["has_tumor"].sum()}')
print(f'  Non-tumor slices: {(~slices_df["has_tumor"]).sum()}')

# ★ FILTER: Only keep slices WITH tumor ★
if TUMOR_ONLY:
    before = len(slices_df)
    slices_df = slices_df[slices_df['has_tumor'] == True].copy()
    print(f'\n★ TUMOR-ONLY FILTER: {before} → {len(slices_df)} slices')
    print(f'  (Removed {before - len(slices_df)} non-tumor slices)')
    print(f'  Rationale: non-tumor slices are identical across stages')
    print(f'  and cause majority-class collapse during training.\n')

# Merge: attach label and split per slice
merged = slices_df.merge(
    splits_df[['PatientID', 'label', 'split']],
    left_on='patient_id', right_on='PatientID', how='inner'
)
merged = merged.drop(columns=['PatientID'])

print(f'After merge: {len(merged)} slices')
print(f'\nSlices per split:')
print(merged['split'].value_counts())
print(f'\nSlices per label:')
print(merged['label'].value_counts().sort_index())

In [ ]:
# ── Save per-split slice CSVs ────────────────────────────────────
for split_name in ['train', 'val', 'test']:
    split_df = merged[merged['split'] == split_name].copy()
    csv_path = os.path.join(OUTPUT_DIR, f'{split_name}_slices.csv')
    split_df.to_csv(csv_path, index=False)
    print(f'{split_name}: {len(split_df)} slices → {csv_path}')
    print(f'  Labels: {dict(split_df["label"].value_counts().sort_index())}')

# Also update slices_index with labels
merged.to_csv(os.path.join(OUTPUT_DIR, 'slices_index_labeled.csv'),
              index=False)
print(f'\n✅ All split CSVs saved to {OUTPUT_DIR}/')

In [ ]:
# ── Visualization ────────────────────────────────────────────────
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Overall class distribution
class_counts = merged['label'].value_counts().sort_index()
labels_plot = [stage_names.get(i, f'Class {i}') for i in class_counts.index]
colors = ['#2196F3', '#4CAF50', '#FF9800', '#F44336'][:len(class_counts)]
axes[0].bar(labels_plot, class_counts.values, color=colors)
axes[0].set_title('Class Distribution (tumor slices only)')
axes[0].set_ylabel('Number of slices')
for i, v in enumerate(class_counts.values):
    axes[0].text(i, v + 5, str(v), ha='center', fontweight='bold')

# Split distribution
split_counts = merged.groupby(['split', 'label']).size().unstack(fill_value=0)
split_counts.plot(kind='bar', ax=axes[1], width=0.8)
axes[1].set_title('Class Distribution per Split')
axes[1].set_ylabel('Number of slices')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=0)
axes[1].legend([stage_names.get(i, f'Class {i}') for i in split_counts.columns],
               loc='upper right')

plt.tight_layout()
plt.show()

# Print imbalance ratio
max_c = class_counts.max()
min_c = class_counts.min()
print(f'\nImbalance ratio: {max_c/min_c:.1f}:1 (max/min class)')
print(f'This will be handled by oversampling + focal loss in training.')